[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/personal-health-agent/personal-health-agent.github.io/blob/main/public/weeks/07-exercise.ipynb)

# Week 7 exercise: wearable time series and image search

Section 2 analyzes four weeks of synthetic data from a [Fitbit Air](https://store.google.com/product/fitbit_air),
Google's screenless tracker, in DuckDB: hourly windows, a personal baseline, and aligning two series. The tables
follow the data types the [Google Health API](https://developers.google.com/health/data-types) returns for the
Fitbit Air. Section 3 searches a few health images with OpenAI, the same way you searched notes in Week 6.

**How to work.** Each question has a `submit("id", """ ... """)` cell: edit the SQL between the triple quotes and
run the cell. It shows the result, checks it, and saves it. Write what you found in the `explain("id", """ ... """)`
cell under it. `run(sql)` runs any query without checking it.

**How to submit.** Submit this notebook, with its outputs: see section 4.

## 0 · Setup

1. In Colab, open the key icon (**Secrets**) in the left sidebar, add `OPENAI_API_KEY`, and turn on notebook
   access. Section 2 needs no key; section 3 calls OpenAI (describing ten images costs about a cent). The key stays
   in your Colab account and is not saved in the notebook.
2. Fill in your name and UNI below, then run the two cells. The second installs DuckDB 1.5.6 and its extensions,
   builds the Fitbit Air tables, and downloads ten images from the course site; it takes about a minute.

In [ ]:
NAME = ""
UNI = ""
OPENAI_MODEL = "gpt-5.6-luna"                 # the vision and chat model
EMBEDDING_MODEL = "text-embedding-3-small"    # 1,536 numbers per vector

In [ ]:
# @title Setup: run this cell (you do not need to read it)
import importlib.metadata, subprocess, sys
def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None
if _version("duckdb") != "1.5.6":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.5.6"], check=True)
import duckdb
if duckdb.__version__ != "1.5.6":
    raise RuntimeError("An older DuckDB is still loaded. Use Runtime > Restart session, then run this cell again.")

"""Helpers for the Week 7 exercise notebook (week07-exercise.ipynb): wearable time series and image search.

build_notebook.py embeds this file and fitbit_air_data.py in the notebook's setup cell, so the notebook runs on its
own in Colab. Edit these files, then rebuild the notebook. Students call only the functions listed under
"Student functions" at the end.

Section 2 uses synthetic Fitbit Air data shaped like the Google Health API's data types; section 3 describes a few
openly licensed health images with an OpenAI vision model and searches the descriptions, through the Flock DuckDB
extension, as in Week 6. For local testing, OPENAI_BASE_URL points Flock at a stand-in server, and IMAGE_DIR
(a folder of images) or IMAGE_URL_BASE (another address for the same files) replaces the course-site download.
"""
import datetime as _dt
import decimal as _decimal
import html as _html
import json
import os
import re

import duckdb
import pandas as pd
from IPython.display import HTML, display

# ---------- keys: Colab secrets first, then environment variables ----------
def _secret(name):
    try:
        from google.colab import userdata  # only in Colab
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name) or None

OPENAI_API_KEY = _secret("OPENAI_API_KEY")
OPENAI_MODEL = globals().get("OPENAI_MODEL") or "gpt-5.6-luna"
EMBEDDING_MODEL = globals().get("EMBEDDING_MODEL") or "text-embedding-3-small"
DIMS = int(globals().get("EMBEDDING_DIMS") or 1536)

# ---------- section 2 data: synthetic Fitbit Air tables (fitbit_air_data.py) ----------
FITBIT_AIR_DATA = '"""Synthetic Fitbit Air data for the Week 7 exercise, shaped like the Google Health API\'s data types.\n\nOne person, 28 days (2026-09-14 to 2026-10-11), local time. Deterministic (seed 46, chosen so the rolling baseline flags the first two sick days, misses the third, and raises one false alarm). The tables follow the\nGoogle Health API data types that list Fitbit Air as a compatible device:\n  heart-rate (Sample)                         -> heart_rate(sample_time, bpm), one sample every 5 seconds while worn\n  steps (Interval)                            -> steps(start_time, end_time, count), one row per minute with steps\n  sleep (Session, with stages)                -> sleep(sleep_id, start_time, end_time, minutes_asleep, minutes_awake)\n                                                 sleep_stages(sleep_id, start_time, end_time, stage)\n  exercise (Session)                          -> exercise(exercise_id, start_time, end_time, activity_type)\n  daily-resting-heart-rate (Daily)            -> daily_resting_heart_rate(date, bpm)\n  daily-heart-rate-variability (Daily)        -> daily_heart_rate_variability(date, rmssd_ms)\n  daily-respiratory-rate (Daily)              -> daily_respiratory_rate(date, breaths_per_minute)\n  daily-sleep-temperature-derivations (Daily) -> daily_sleep_temperature(date, relative_c)\nBuilt into the story: the band charges every morning (no heart rate for about an hour), the person forgets to wear it\none night (no sleep record), runs three evenings a week, and is ill from Oct 2 to Oct 4 (resting heart rate and\nbreathing rate up, HRV down, skin temperature up, fewer steps).\n"""\nimport datetime as dt\n\nimport numpy as np\nimport pandas as pd\n\nSTART = dt.date(2026, 9, 14)\nDAYS = 28\nILL = {dt.date(2026, 10, 2), dt.date(2026, 10, 3), dt.date(2026, 10, 4)}\nFORGOT_NIGHT = dt.date(2026, 9, 22)          # band left on the charger overnight: no sleep session that night\nRUN_DAYS = {0, 2, 5}                          # Monday, Wednesday, Saturday\n\n\ndef generate(seed=46):\n    rng = np.random.default_rng(seed)\n    minutes = pd.date_range(dt.datetime.combine(START, dt.time()), periods=DAYS * 1440, freq="min")\n    steps_min = np.zeros(len(minutes), dtype=int)\n    days = [START + dt.timedelta(days=d) for d in range(DAYS)]\n    ill = [day in ILL for day in days]\n    rhr = [58 + rng.normal(0, 2.0) + (8 if i else 0) for i in ill]\n    daily = [(day, round(r), round(52 * (0.68 if i else 1) + rng.normal(0, 3), 1),\n              round(14.5 + rng.normal(0, 0.3) + (2.4 if i else 0), 1), round(rng.normal(0, 0.15) + (0.9 if i else 0), 2))\n             for day, r, i in zip(days, rhr, ill)]\n    # 1. Awake heart rate for every minute, around the day\'s resting rate + 14; sleep, activity, and gaps overwrite it.\n    hr_min = np.repeat(np.array(rhr) + 14, 1440) + rng.normal(0, 4, len(minutes))\n    sleeps, stages, exercises, gaps = [], [], [], []\n    for d, day in enumerate(days):\n        base = d * 1440\n        # Walking bouts during the day (fewer while ill).\n        for _ in range(rng.integers(5, 9) // (3 if ill[d] else 1)):\n            s = base + int(rng.integers(8 * 60, 21 * 60)); n = int(rng.integers(5, 25))\n            steps_min[s:s + n] = rng.integers(70, 115, n); hr_min[s:s + n] += rng.normal(18, 4, n)\n        # Runs: about 18:00 for 40 minutes on run days, not while ill.\n        if day.weekday() in RUN_DAYS and not ill[d]:\n            s = base + 18 * 60 + int(rng.integers(0, 15)); n = 40\n            hr_min[s:s + n] = np.linspace(110, 158, n) + rng.normal(0, 3, n)\n            steps_min[s:s + n] = rng.integers(150, 172, n)\n            exercises.append((f"E{len(exercises) + 1:02d}", minutes[s], minutes[s + n], "RUNNING"))\n        # Charging every morning from about 08:00, for about an hour: no heart rate.\n        c = base + 8 * 60 + int(rng.integers(0, 30)); gaps.append((c, c + int(rng.integers(50, 75))))\n        # The night from about 23:00 to about 07:00 the next morning. The data ends at midnight after the last day.\n        if d == DAYS - 1:\n            continue\n        night_start = base + 23 * 60 + int(rng.integers(-20, 30))\n        night_end = base + 1440 + 7 * 60 + int(rng.integers(-15, 25))\n        if day == FORGOT_NIGHT:                  # the band stayed on the charger overnight: no readings, no sleep session\n            gaps.append((night_start, night_end))\n            continue\n        t, sid, seg = night_start, f"S{len(sleeps) + 1:02d}", []\n        while t < night_end:                     # 90-minute cycles: light, deep (early in the night), light, REM, brief awake\n            early = (t - night_start) < 240\n            for stage, (lo, hi) in [("LIGHT", (20, 35)), ("DEEP", (15, 30) if early else (0, 8)), ("LIGHT", (10, 20)),\n                                    ("REM", (8, 15) if early else (18, 30)), ("AWAKE", (0, 4))]:\n                n = int(rng.integers(lo, hi + 1))\n                if n == 0 or t >= night_end:\n                    continue\n                n = min(n, night_end - t); seg.append((stage, t, t + n)); t += n\n        offset = {"DEEP": -6, "LIGHT": -3, "REM": 2, "AWAKE": 8}\n        for stage, a, b in seg:\n            hr_min[a:b] = rhr[d] + offset[stage] + rng.normal(0, 1.5, b - a)\n            stages.append((sid, minutes[a], minutes[b - 1] + pd.Timedelta(minutes=1), stage))\n        awake = sum(b - a for s_, a, b in seg if s_ == "AWAKE")\n        sleeps.append((sid, minutes[night_start], minutes[night_end - 1] + pd.Timedelta(minutes=1),\n                       night_end - night_start - awake, awake))\n    for a, b in gaps:\n        hr_min[a:b] = np.nan\n    # Heart rate samples every 5 seconds from the minute values.\n    worn = ~np.isnan(hr_min)\n    hr_t = (minutes[worn].values[:, None] + (np.arange(12) * np.timedelta64(5, "s"))[None, :]).ravel()\n    hr_v = (hr_min[worn][:, None] + rng.normal(0, 1.2, (worn.sum(), 12))).ravel()\n    heart_rate = pd.DataFrame({"sample_time": hr_t, "bpm": np.round(hr_v).astype(int)})\n    step_rows = steps_min > 0\n    steps = pd.DataFrame({"start_time": minutes[step_rows], "end_time": minutes[step_rows] + pd.Timedelta(minutes=1),\n                          "count": steps_min[step_rows]})\n    tables = {\n        "heart_rate": heart_rate,\n        "steps": steps,\n        "sleep": pd.DataFrame(sleeps, columns=["sleep_id", "start_time", "end_time", "minutes_asleep", "minutes_awake"]),\n        "sleep_stages": pd.DataFrame(stages, columns=["sleep_id", "start_time", "end_time", "stage"]),\n        "exercise": pd.DataFrame(exercises, columns=["exercise_id", "start_time", "end_time", "activity_type"]),\n        "daily_resting_heart_rate": pd.DataFrame([(r[0], r[1]) for r in daily], columns=["date", "bpm"]),\n        "daily_heart_rate_variability": pd.DataFrame([(r[0], r[2]) for r in daily], columns=["date", "rmssd_ms"]),\n        "daily_respiratory_rate": pd.DataFrame([(r[0], r[3]) for r in daily], columns=["date", "breaths_per_minute"]),\n        "daily_sleep_temperature": pd.DataFrame([(r[0], r[4]) for r in daily], columns=["date", "relative_c"]),\n    }\n    return tables\n\n\nif __name__ == "__main__":\n    for name, df in generate().items():\n        print(f"{name}: {len(df)} rows; columns {list(df.columns)}")\n'
_ns = {"__name__": "fitbit_air_data"}
exec(FITBIT_AIR_DATA, _ns)
TABLE_INFO = {   # table -> Google Health API data type it follows
    "heart_rate": "heart-rate (Sample: one reading every 5 seconds while the band is worn)",
    "steps": "steps (Interval: one row per minute with steps)",
    "sleep": "sleep (Session)",
    "sleep_stages": "sleep stages (part of the sleep Session: LIGHT, DEEP, REM, AWAKE)",
    "exercise": "exercise (Session)",
    "daily_resting_heart_rate": "daily-resting-heart-rate (Daily)",
    "daily_heart_rate_variability": "daily-heart-rate-variability (Daily: RMSSD in milliseconds)",
    "daily_respiratory_rate": "daily-respiratory-rate (Daily: breaths per minute during sleep)",
    "daily_sleep_temperature": "daily-sleep-temperature-derivations (Daily: skin temperature relative to baseline, °C)",
}
con = duckdb.connect()
for _name, _df in _ns["generate"]().items():
    con.register("_df", _df)
    con.execute(f"CREATE TABLE {_name} AS SELECT * FROM _df")
    con.unregister("_df")
con.execute("INSTALL vss; LOAD vss;")

# ---------- section 3 data: openly licensed images from Wikimedia Commons, hosted on the course site ----------
IMAGES = [{'image_id': 'fundus_normal', 'shows': 'a photograph of a normal retina (left eye)', 'category': 'eye', 'url': 'https://personal-health-agent.github.io/weeks/07-images/fundus_normal.jpg', 'credit': 'Mikael Häggström', 'license': 'CC0', 'page': 'https://commons.wikimedia.org/wiki/File:Fundus_photograph_of_normal_left_eye.jpg'}, {'image_id': 'fundus_retinopathy', 'shows': 'a retinal photograph showing retinopathy', 'category': 'eye', 'url': 'https://personal-health-agent.github.io/weeks/07-images/fundus_retinopathy.jpg', 'credit': 'National Eye Institute', 'license': 'Public domain', 'page': 'https://commons.wikimedia.org/wiki/File:Fundus_retinopathy_EDA03.JPG'}, {'image_id': 'chest_xray', 'shows': 'a normal chest X-ray', 'category': 'x-ray', 'url': 'https://personal-health-agent.github.io/weeks/07-images/chest_xray.jpg', 'credit': 'Mikael Häggström', 'license': 'CC0', 'page': 'https://commons.wikimedia.org/wiki/File:Normal_posteroanterior_(PA)_chest_radiograph_(X-ray).jpg'}, {'image_id': 'hives', 'shows': 'hives (cold urticaria) on the skin', 'category': 'skin', 'url': 'https://personal-health-agent.github.io/weeks/07-images/hives.jpg', 'credit': 'Rylan7667', 'license': 'CC0', 'page': 'https://commons.wikimedia.org/wiki/File:Cold_Urticaria.jpg'}, {'image_id': 'stitches', 'shows': 'a wound closed with stitches', 'category': 'skin', 'url': 'https://personal-health-agent.github.io/weeks/07-images/stitches.jpg', 'credit': 'Wompwomp', 'license': 'CC BY-SA 3.0', 'page': 'https://commons.wikimedia.org/wiki/File:12_Stitches.jpg'}, {'image_id': 'meal_spaghetti', 'shows': 'a plate of spaghetti', 'category': 'meal', 'url': 'https://personal-health-agent.github.io/weeks/07-images/meal_spaghetti.jpg', 'credit': 'Unknown author', 'license': 'Public domain', 'page': 'https://commons.wikimedia.org/wiki/File:Spaghetti-prepared.jpg'}, {'image_id': 'meal_pancakes', 'shows': 'thin pancakes with strips of bacon', 'category': 'meal', 'url': 'https://personal-health-agent.github.io/weeks/07-images/meal_pancakes.jpg', 'credit': "CharmaineZoe's Marvelous Melange", 'license': 'CC BY 2.0', 'page': 'https://commons.wikimedia.org/wiki/File:Breakfast_Pancakes_with_maple_syrup_%26_bacon_(15544231734).jpg'}, {'image_id': 'meal_salad', 'shows': 'a Greek salad with feta, cucumber, tomato, pepper, and olives', 'category': 'meal', 'url': 'https://personal-health-agent.github.io/weeks/07-images/meal_salad.jpg', 'credit': 'Jpatokal', 'license': 'CC BY-SA 4.0', 'page': 'https://commons.wikimedia.org/wiki/File:Greece_Food_Horiatiki.JPG'}, {'image_id': 'glucose_meter', 'shows': 'a device for checking blood glucose', 'category': 'device', 'url': 'https://personal-health-agent.github.io/weeks/07-images/glucose_meter.jpg', 'credit': 'Tomwsulcer', 'license': 'CC0', 'page': 'https://commons.wikimedia.org/wiki/File:Device_to_check_for_diabetes_2.jpg'}, {'image_id': 'pill_bottle', 'shows': 'a bottle of prescription tablets (Stribild)', 'category': 'medication', 'url': 'https://personal-health-agent.github.io/weeks/07-images/pill_bottle.jpg', 'credit': 'JörgenMoorlag', 'license': 'CC BY-SA 4.0', 'page': 'https://commons.wikimedia.org/wiki/File:Stribild_bottle_Dutch_labeling.jpg'}]
IMAGE_DIR = os.environ.get("IMAGE_DIR") or "images"

def _fetch_images():
    "Download each image once, as PNG (Flock sends images to OpenAI labeled as PNG)."
    import time, urllib.error, urllib.request
    from PIL import Image
    os.makedirs(IMAGE_DIR, exist_ok=True)
    rows, failed = [], []
    for im in IMAGES:
        path = os.path.join(IMAGE_DIR, im["image_id"] + ".png")
        wait = 5
        for attempt in range(6):
            if os.path.exists(path):
                break
            try:   # IMAGE_URL_BASE (testing only) serves the same files from another address
                url = os.environ["IMAGE_URL_BASE"].rstrip("/") + "/" + im["url"].rsplit("/", 1)[1] if os.environ.get("IMAGE_URL_BASE") else im["url"]
                req = urllib.request.Request(url, headers={"User-Agent": "BINF4070-course-exercise/1.0 (Columbia University)"})
                with urllib.request.urlopen(req, timeout=30) as resp, open(path + ".download", "wb") as f:
                    f.write(resp.read())
                Image.open(path + ".download").convert("RGB").save(path)
                os.remove(path + ".download")
                time.sleep(1.5)
            except urllib.error.HTTPError as e:
                if e.code != 429:
                    break
                retry_after = e.headers.get("Retry-After", "")
                print(f"The server asked to slow down; waiting {retry_after if retry_after.isdigit() else wait} s before {im['image_id']}.")
                time.sleep(int(retry_after) if retry_after.isdigit() else wait)
                wait = min(wait * 2, 60)
            except Exception:
                time.sleep(wait)
                wait = min(wait * 2, 60)
        if os.path.exists(path):
            rows.append((im["image_id"], os.path.abspath(path), im["shows"], im["category"], im["credit"], im["license"], im["page"]))
        else:
            failed.append(im["image_id"])
    con.execute("CREATE OR REPLACE TABLE images (image_id VARCHAR, path VARCHAR, shows VARCHAR, category VARCHAR, "
                "credit VARCHAR, license VARCHAR, source VARCHAR)")
    con.executemany("INSERT INTO images VALUES (?, ?, ?, ?, ?, ?, ?)", rows)
    if failed:
        print(f"Could not download {', '.join(failed)}; run the setup cell again to retry. Images already downloaded are kept.")

_fetch_images()
IMAGE_INFO = {i: (shows, cat) for i, shows, cat in con.sql("SELECT image_id, shows, category FROM images").fetchall()}

# ---------- Flock: SQL functions that call OpenAI ----------
# The embedding model sends up to 64 texts per request. The chat model sends one row per request, so its
# call count equals the rows sent. usage_limit caps each model's tokens for the session.
BUDGET = {"total_tokens_limit": 300_000}
FLOCK_MODELS = {
    "embed": (EMBEDDING_MODEL, {"max_batch_size": 64, "usage_limit": {"total_tokens_limit": 500_000}}),
    "luna": (OPENAI_MODEL, {"max_batch_size": 1, "usage_limit": BUDGET}),
}

# Flock's Linux build looks for CA certificates at /etc/ssl/cert.pem and the Red Hat path below. Ubuntu, which
# Colab runs, keeps them at /etc/ssl/certs/ca-certificates.crt, so without these links every HTTPS call fails
# with "Empty streaming response from provider (HTTP 0)". Colab runs as root, so the links can be made.
def _link_ca_bundle():
    import platform
    bundle = "/etc/ssl/certs/ca-certificates.crt"
    if platform.system() != "Linux" or not os.path.exists(bundle):
        return
    os.environ.setdefault("SSL_CERT_FILE", bundle)
    for path in ("/etc/ssl/cert.pem", "/etc/pki/tls/certs/ca-bundle.crt"):
        if not os.path.exists(path):
            try:
                os.makedirs(os.path.dirname(path), exist_ok=True)
                os.symlink(bundle, path)
            except OSError as e:
                print(f"Could not link {path} to {bundle} ({e}); Flock's model calls may fail with HTTP 0.")

_link_ca_bundle()
con.execute("INSTALL flock FROM community; LOAD flock;")
if OPENAI_API_KEY:
    _base = os.environ.get("OPENAI_BASE_URL")
    _q = lambda v: "'" + str(v).replace("'", "''") + "'"
    con.execute(f"CREATE OR REPLACE SECRET (TYPE OPENAI, API_KEY {_q(OPENAI_API_KEY)}" + (f", BASE_URL {_q(_base)}" if _base else "") + ")")
# The models are defined even without a key, so a query fails on the missing key rather than on the model.
for _name, (_model, _args) in FLOCK_MODELS.items():
    _stmt = f"MODEL('{_name}', '{_model}', 'openai', {json.dumps(_args)})"
    try:
        con.execute("CREATE " + _stmt)
    except duckdb.Error:
        con.execute("UPDATE " + _stmt)   # the model already exists in this runtime

# Shortcuts, so students write short SQL. Each wraps one Flock call.
DESCRIBE_PROMPT = ("Describe this image in one or two sentences, as a caption for searching a personal health record: "
                   "say what it shows and anything health-related in it.")
# DuckDB evaluates a function early, while planning, when all its inputs are constants, and it may do so several
# times. Passing each shortcut's input through at_run_time(), which DuckDB must run during the query, makes every
# call happen once per row, at run time.
NO_KEY = ("This needs an OpenAI key: add OPENAI_API_KEY in the Colab Secrets panel (the key icon), turn on notebook "
          "access, and run the setup cell again.")
con.create_function("at_run_time", lambda text: text, ["VARCHAR"], "VARCHAR", side_effects=True)
for _macro in (
    f"embed(text) AS llm_embedding({{'model_name': 'embed'}}, {{'context_columns': [{{'data': at_run_time(text)}}]}})::FLOAT[{DIMS}]",
    f"describe_image(path) AS llm_complete({{'model_name': 'luna'}}, {{'prompt': '{DESCRIBE_PROMPT}', "
    "'context_columns': [{'data': at_run_time(path), 'type': 'image'}]})::VARCHAR",
):
    if OPENAI_API_KEY:
        con.execute("CREATE OR REPLACE MACRO " + _macro)
    else:   # Flock checks for its OpenAI secret when a macro is defined, so without a key each shortcut says what is missing
        con.execute(f"CREATE OR REPLACE MACRO {_macro.split(' AS ', 1)[0]} AS error('{NO_KEY}')")

def _flock_metrics():
    "Model calls, tokens, and models used per Flock function since the last reset."
    out = {}
    for key, m in json.loads(con.sql("SELECT flock_get_metrics()").fetchone()[0] or "{}").items():
        fn = re.sub(r"_\d+$", "", key)
        s = out.setdefault(fn, {"calls": 0, "input_tokens": 0, "output_tokens": 0, "models": set()})
        s["calls"] += m.get("api_calls", 0)
        s["input_tokens"] += m.get("input_tokens", 0)
        s["output_tokens"] += m.get("output_tokens", 0)
        s["models"].add(m.get("model_name"))
    return out

# ---------- questions (from build_notebook.py) and saved work ----------
QUESTIONS = {'2.1': {'label': 'Hourly windows', 'question': "The query computes each hour's mean heart rate on Oct 1 with time_bucket (a tumbling window). Add the number of readings in each hour. A fully worn hour has 720 readings, one every 5 seconds. Which hours have fewer, and why?"}, '2.2': {'label': 'Personal baseline', 'question': "Each day's baseline is the mean resting heart rate of the 7 days before it: the window w slides one day at a time. Add z, how many standard deviations the day is from its baseline, and keep only the days with |z| > 2. Which days stand out?"}, '2.3': {'label': 'Align two series', 'question': "Heart rate and sleep stages have no timestamps in common. ASOF JOIN gives each heart-rate reading the stage that started at or before it, and WHERE keeps only readings before that stage ended. Change the query to return each stage's mean heart rate and number of readings."}, '3.1': {'label': 'Describe the images', 'question': "OpenAI's embedding models take text only, so a vision model first describes each image, and the description is embedded. Run the first cell to describe one image. Then, in the second cell, describe every image into image_captions."}, '3.2': {'label': 'Search with text', 'question': 'Embed each caption into image_vectors, as you embedded the notes in Week 6. Then search the images with text in the second cell, and try other searches: “a skin problem”, “an eye exam”, “a meal for someone watching their blood sugar”.'}, '3.3': {'label': 'Search with an image', 'question': 'Find the 3 images most similar to the normal retinal photo, fundus_normal, using its stored vector instead of a text query. Fill in WHERE: a is fundus_normal, and b is any other image.'}}
WORK_FILE = "week07_work.json"
try:
    with open(WORK_FILE) as _f:
        work = json.load(_f)
except (OSError, ValueError):
    work = {"student": {}, "answers": {}, "history": []}

def _save():
    with open(WORK_FILE, "w") as f:
        json.dump(work, f, default=str, indent=1)

def _record(qid, action, **data):
    t = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")
    work["history"].append({"t": t, "question": qid, "action": action, **data})
    a = work["answers"].setdefault(qid, {})
    if action == "submit":
        # The latest run replaces the saved SQL, result, and check together, so a failed or unchecked run
        # never leaves an earlier run's result or pass status attached to the new SQL.
        a["submissions"] = a.get("submissions", 0) + 1
        for k in ("sql", "result", "check", "error", "model_calls"):
            if k in data:
                a[k] = data[k]
            else:
                a.pop(k, None)
    a["updated_at"] = t
    _save()

def _need(qid):
    if qid not in QUESTIONS:
        raise ValueError(f"Unknown question {qid!r}. Use one of: {', '.join(QUESTIONS)}")

# ---------- display ----------
_COLORS = {"ok": "#e8f2ea", "bad": "#f8e9e9", "note": "#fff4e0", "info": "#e8eef5"}

def _box(kind, text):
    display(HTML(f'<div style="background:{_COLORS[kind]};border-radius:6px;padding:8px 12px;margin:6px 0;'
                 f'color:#1a1a1a;white-space:pre-wrap">{_html.escape(text)}</div>'))

def _short(v):
    "Long vectors are shown as their first values and their length."
    if isinstance(v, (list, tuple)) and len(v) > 6:
        return "[" + ", ".join(f"{x:.3f}" if isinstance(x, float) else str(x) for x in v[:4]) + f", … ({len(v)} values)]"
    return v

def _frame(cols, rows, limit=20):
    df = pd.DataFrame([[_short(v) for v in r] for r in rows[:limit]], columns=cols)
    with pd.option_context("display.max_colwidth", 120):
        display(df)
    print(f"... {len(rows)} rows in total; showing the first {limit}." if len(rows) > limit else f"{len(rows)} row(s)")

class Result:
    "One submitted statement: its output rows, the table it created, and the model calls it made."
    def __init__(self, sql, cols, rows, table, flock):
        self.sql, self.cols, self.table, self.flock = sql, list(cols or []), table, flock
        seen = {}
        for k, c in enumerate(self.cols):
            seen[c] = seen.get(c, 0) + 1
            if seen[c] > 1:
                self.cols[k] = f"{c}_{seen[c]}"   # SELECT a.note_id, b.note_id returns two columns named note_id
        # DECIMAL values (for example from ROUND on a literal) arrive as Decimal; checks compare floats.
        self.rows = [{c: float(v) if isinstance(v, _decimal.Decimal) else v for c, v in zip(self.cols, r)} for r in (rows or [])]

    def calls(self, fn):
        return self.flock.get(fn, {}).get("calls", 0)

def _summary(r):
    rows = [{k: _short(v) for k, v in x.items()} for x in r.rows[:10]]
    return {"columns": r.cols, "row_count": len(r.rows), "rows": json.loads(json.dumps(rows, default=str))}

_CREATE_AS = re.compile(r'CREATE\s+(?:OR\s+REPLACE\s+)?(?:TEMP(?:ORARY)?\s+)?TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?("[^"]+"|[\w.]+)\s+AS\b', re.I)

def _execute(sql):
    "Run a statement; if it created a table with CREATE TABLE ... AS, return that table's rows."
    if OPENAI_API_KEY:
        con.execute("SELECT flock_reset_metrics()")
    rel = con.sql(sql)
    cols, rows, table = (rel.columns, rel.fetchall(), None) if rel is not None else (None, None, None)
    flock = _flock_metrics() if OPENAI_API_KEY else {}
    if cols is None:
        made = _CREATE_AS.findall(sql)
        if made:
            table = made[-1].strip('"')
            rel = con.sql(f'SELECT * FROM {made[-1]}')
            cols, rows = rel.columns, rel.fetchall()
    return Result(sql, cols, rows, table, flock)

def _meter(r):
    return "; ".join(f"{fn}: {s['calls']} request(s), {s['input_tokens']} input and {s['output_tokens']} output tokens "
                     f"({', '.join(sorted(m for m in s['models'] if m))})" for fn, s in r.flock.items())

def _model_calls(r):
    return {fn: {"calls": s["calls"], "input_tokens": s["input_tokens"], "output_tokens": s["output_tokens"],
                 "models": sorted(m for m in s["models"] if m)} for fn, s in r.flock.items()}

# ---------- checks: shared pieces ----------
OK = lambda m, show=None: ("ok", m, show)
BAD = lambda m: ("bad", m, None)

def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)

def _near(a, b, tol=0.05):
    return _num(a) and abs(a - b) < tol

def _col(r, test, skip=()):
    return next((c for c in r.cols if c not in skip and any(x[c] is not None for x in r.rows)
                 and all(x[c] is None or test(x[c]) for x in r.rows)), None)

def _table_info(name):
    try:
        cols = con.sql(f"DESCRIBE {name}").fetchall()
    except duckdb.Error:
        return None
    vec = next(((c, t) for c, t, *_ in cols if re.fullmatch(r"FLOAT\[\d+\]", t)), None)
    return {"rows": con.sql(f"SELECT COUNT(*) FROM {name}").fetchone()[0], "cols": [c for c, *_ in cols],
            "vec": vec[0] if vec else None, "types": {c: t for c, t, *_ in cols}}

# ---------- section 2: time series ----------
DAY = "2026-10-01"
FULL_HOUR = 720   # one reading every 5 seconds

def _hourly_expected():
    return {h: (m, n) for h, m, n in con.sql(f"""SELECT time_bucket(INTERVAL 1 HOUR, sample_time) AS h, avg(bpm), count(*)
        FROM heart_rate WHERE sample_time >= TIMESTAMP '{DAY}' AND sample_time < TIMESTAMP '{DAY}' + INTERVAL 1 DAY
        GROUP BY h""").fetchall()}

def _check_2_1(r):
    want = _hourly_expected()
    hc = _col(r, lambda v: isinstance(v, _dt.datetime))
    if not hc:
        return BAD("Keep the hour column from time_bucket in the result.")
    rows = [x for x in r.rows if x[hc] in want]
    if len(rows) != len(want) or len(r.rows) != len(want):
        return BAD(f"Expected one row for each hour of {DAY} with readings ({len(want)} rows); your result has {len(r.rows)}.")
    mean = next((c for c in r.cols if all(_near(x[c], want[x[hc]][0]) for x in rows)), None)
    count = next((c for c in r.cols if all(x[c] == want[x[hc]][1] for x in rows)), None)
    if not mean:
        return BAD("Keep the hourly mean heart rate, avg(bpm).")
    if not count:
        return BAD("Add the number of readings in each hour: count(*) AS n.")
    short = [f"{h:%H}:00 ({n})" for h, (_, n) in sorted(want.items()) if n < 0.9 * FULL_HOUR]
    return OK(f"Correct. A fully worn hour has {FULL_HOUR} readings; these hours have fewer: {', '.join(short)}. "
              "What happened then, and how much should you trust those hours' means?")

def _z_expected():
    return {d: (z, bpm) for d, bpm, z in con.sql("""SELECT date, bpm, (bpm - avg(bpm) OVER w) / stddev_samp(bpm) OVER w
        FROM daily_resting_heart_rate WINDOW w AS (ORDER BY date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)""").fetchall()}

def _check_2_2(r):
    want = _z_expected()
    flagged = sorted(d for d, (z, _) in want.items() if z is not None and abs(z) > 2)
    dc = _col(r, lambda v: isinstance(v, _dt.date) and not isinstance(v, _dt.datetime))
    if not dc:
        return BAD("Keep the date column in the result.")
    zc = next((c for c in r.cols if r.rows and all(_near(x[c], want[x[dc]][0], 0.01) for x in r.rows if want[x[dc]][0] is not None)
               and any(x[c] is not None for x in r.rows) and c != "bpm"), None)
    days = sorted(x[dc] for x in r.rows)
    if days == flagged:
        ill = {_dt.date(2026, 10, 2), _dt.date(2026, 10, 3), _dt.date(2026, 10, 4)}
        missed = [f"{d:%b %d}" for d in sorted(ill) if d not in flagged]
        other = [f"{d:%b %d}" for d in flagged if d not in ill]
        return OK("Correct: " + ", ".join(f"{d:%b %d} (z = {want[d][0]:.1f})" for d in flagged) +
                  ". The person was ill from Oct 2 to Oct 4." + (f" Why was {', '.join(missed)} missed?" if missed else "") +
                  (f" What is {', '.join(other)}?" if other else "") + " Check the other daily tables (HRV, breathing "
                  "rate, skin temperature) for the same days with run().")
    if not zc:
        return BAD("Add z = (bpm − the 7-day mean) / the 7-day standard deviation, using the same window w for both.")
    return BAD("The z column is right. Now keep only the days with |z| > 2, for example with QUALIFY abs(z) > 2.")

def _stage_expected():
    return {st: m for st, m in con.sql("""SELECT s.stage, avg(h.bpm) FROM heart_rate AS h ASOF JOIN sleep_stages AS s
        ON h.sample_time >= s.start_time WHERE h.sample_time < s.end_time GROUP BY s.stage""").fetchall()}

def _check_2_3(r):
    want = _stage_expected()
    sc = _col(r, lambda v: isinstance(v, str) and v in want)
    if not sc or len(r.rows) != len(want):
        return BAD("Return one row per sleep stage (LIGHT, DEEP, REM, AWAKE): GROUP BY s.stage.")
    mean = next((c for c in r.cols if all(_near(x[c], want[x[sc]]) for x in r.rows)), None)
    if not mean:
        if "end_time" not in r.sql.lower():
            return BAD("The means are off. Keep WHERE h.sample_time < s.end_time: without it, every daytime reading is "
                       "matched to the last stage of the night before.")
        return BAD("Expected each stage's mean heart rate: avg(h.bpm).")
    order = " < ".join(f"{st} {m:.1f}" for st, m in sorted(want.items(), key=lambda kv: kv[1]))
    return OK(f"Correct: {order} beats per minute. Why is an ASOF JOIN needed here rather than an equality join on time?")

# ---------- section 3: images ----------
def _show_images(ids, sims=None):
    "Thumbnails of the given images, in order, with what each shows."
    import base64
    cells = []
    for k, i in enumerate(ids):
        path = os.path.join(IMAGE_DIR, i + ".png")
        if not os.path.exists(path):
            continue
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        sim = f"<br>similarity {sims[k]:.3f}" if sims and sims[k] is not None else ""
        cells.append(f'<div style="width:170px;font-size:12px"><img src="data:image/png;base64,{b64}" '
                     f'style="width:160px;height:120px;object-fit:cover"><br><b>{_html.escape(i)}</b>{sim}<br>'
                     f'{_html.escape(IMAGE_INFO.get(i, ("", ""))[0])}</div>')
    display(HTML('<div style="display:flex;gap:10px;flex-wrap:wrap">' + "".join(cells) + "</div>"))

def _image_ids(r):
    c = _col(r, lambda v: isinstance(v, str) and v in IMAGE_INFO)
    return [x[c] for x in r.rows] if c else None

def _check_3_1(r):
    t = _table_info("image_captions")
    if t is None:
        if r.calls("llm_complete"):
            return OK("One caption. Compare it with the image; then create image_captions for every image in the next cell.")
        return BAD("This question needs describe_image(path).")
    n = con.sql("SELECT COUNT(*) FROM images").fetchone()[0]
    if t["rows"] != n or "caption" not in t["cols"]:
        return BAD(f"image_captions should have image_id and caption for all {n} images; it has {t['rows']} rows.")
    if con.sql("SELECT COUNT(*) FROM image_captions WHERE caption IS NULL OR length(caption) < 10").fetchone()[0]:
        return BAD("Some captions are empty. Run the cell again.")
    rows = con.sql("SELECT c.image_id, i.shows, c.caption FROM image_captions c JOIN images i USING (image_id) ORDER BY 1").fetchall()
    def show():
        with pd.option_context("display.max_colwidth", 160):
            display(pd.DataFrame(rows, columns=["image_id", "what the image shows (labeled by hand)", "model's caption"]))
    return OK(f"{n} captions written. Compare each with what the image shows: which captions miss or invent a "
              "health-related detail?", show)

def _check_3_2(r):
    t = _table_info("image_vectors")
    if t is None or not t["vec"]:
        return BAD("Create image_vectors with a fixed-length vector column: embed(caption).")
    if r.table == "image_vectors" or not r.calls("llm_embedding") or _image_ids(r) is None:
        n = con.sql("SELECT COUNT(*) FROM images").fetchone()[0]
        if t["rows"] != n:
            return BAD(f"image_vectors should have one vector per image ({n}); it has {t['rows']}.")
        return OK(f"image_vectors holds {n} vectors. Now search them with a text query in the next cell.")
    ids = _image_ids(r)
    sims = [x.get("similarity") for x in r.rows]
    cats = [IMAGE_INFO[i][1] for i in ids]
    return OK(f"Top results: {', '.join(f'{i} ({c})' for i, c in zip(ids, cats))}. Try other searches: do the results "
              "match what you asked for?", lambda: _show_images(ids, sims))

def _check_3_3(r):
    ids = _image_ids(r)
    if ids is None:
        return BAD("Return the image_id of the most similar images.")
    if "fundus_normal" in ids:
        return BAD("Leave out the image itself: add b.image_id <> a.image_id.")
    sims = [x.get("similarity") for x in r.rows]
    eyes = [i for i in ids if IMAGE_INFO[i][1] == "eye"]
    return OK(f"The most similar images to fundus_normal: {', '.join(ids)}" + (" (the other retinal photo ranks first)."
              if ids and ids[0] in eyes else ".") + " The search compares captions, not pixels: what could two "
              "retinal photos differ in that their captions would not show?", lambda: _show_images(ids, sims))

CHECKS = {"2.1": _check_2_1, "2.2": _check_2_2, "2.3": _check_2_3, "3.1": _check_3_1, "3.2": _check_3_2, "3.3": _check_3_3}

# ---------- Student functions ----------
def show_table(name, n=5):
    "Show a table's row count and its first n rows."
    total = con.sql(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    rel = con.sql(f"SELECT * FROM {name} LIMIT {int(n)}")
    print(f"{name}: {total} rows in total" + (f"; showing the first {n}" if total > n else ""))
    _frame(rel.columns, rel.fetchall(), n)

def submit(qid, sql):
    "Run your SQL for a question, show the result, check it, and save it."
    _need(qid)
    try:
        r = _execute(sql)
    except Exception as e:   # DuckDB errors, and errors Flock raises from its model calls
        msg = f"{type(e).__name__}: {e}"
        if not OPENAI_API_KEY and re.search(r"llm_|Secret not found", str(e)):
            msg = "Flock needs an OpenAI key: add OPENAI_API_KEY in Colab's Secrets panel, run the setup cell again, then this cell.\n" + msg
        elif "Empty streaming response" in str(e) or "HTTP 0" in str(e):
            msg = ("The request never reached OpenAI (no HTTP response), so this is a connection problem, not your SQL. "
                   "Run the setup cell again, then this cell; if it persists, tell the course staff.\n" + msg)
        elif "Output extraction error" in str(e) or "Failed to parse" in str(e):
            msg = ("OpenAI sent back an error instead of an answer. Check that OPENAI_API_KEY is valid and that your key "
                   "can use the model.\n" + msg)
        elif re.search(r"Cannot cast list with length|Array size", str(e)):
            msg = "Two vectors of different lengths were compared, or a list was cast to the wrong FLOAT[n]. Check the dimensions.\n" + msg
        _box("bad", msg)
        _record(qid, "submit", sql=sql, error=str(e))
        return
    if r.table:
        print(f"Created table {r.table}:")
    if r.cols:
        _frame(r.cols, [tuple(x.values()) for x in r.rows])
    else:
        print("Done.")
    if _meter(r):
        print(_meter(r))
    verdict = None
    if qid in CHECKS:
        try:
            status, message, show = CHECKS[qid](r)
        except Exception as e:
            status, message, show = "bad", f"Could not check this result: {e}", None
        _box(status, f"Question {qid}: {message}")
        if show:
            show()
        verdict = {"status": status, "message": message}
    _record(qid, "submit", sql=sql, result=_summary(r), model_calls=_model_calls(r), **({"check": verdict} if verdict else {}))

def run(sql):
    "Run any query and show its result, without checking it. The query is saved in your history."
    try:
        r = _execute(sql)
    except Exception as e:
        _box("bad", f"{type(e).__name__}: {e}")
        work["history"].append({"t": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"),
                                "question": None, "action": "run", "sql": sql, "error": str(e)})
        _save()
        return
    if r.table:
        print(f"Created table {r.table}:")
    if r.cols:
        _frame(r.cols, [tuple(x.values()) for x in r.rows])
    else:
        print("Done.")
    if _meter(r):
        print(_meter(r))
    work["history"].append({"t": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"), "question": None,
                            "action": "run", "sql": sql, "result": _summary(r), "model_calls": _model_calls(r)})
    _save()

def explain(qid, text):
    "Save your written answer for a question."
    _need(qid)
    work["answers"].setdefault(qid, {})["answer"] = text.strip()
    _record(qid, "answer", answer=text.strip())
    print(f"Saved your answer for question {qid}.")

def show_schema():
    "Show each table's columns and the Google Health API data type it follows."
    rows = con.sql("""SELECT table_name, string_agg(column_name || ' ' || data_type, ', ' ORDER BY column_index)
        FROM duckdb_columns() WHERE NOT internal AND schema_name = 'main' GROUP BY table_name""").fetchall()
    order = list(TABLE_INFO) + ["images"]
    rows = sorted(rows, key=lambda x: order.index(x[0]) if x[0] in order else 99)
    with pd.option_context("display.max_colwidth", 120):
        display(pd.DataFrame([(t, c, TABLE_INFO.get(t, "")) for t, c in rows], columns=["table", "columns", "Google Health API data type"]))

def show_images(ids=None):
    "Show thumbnails of the images (all of them, or the given image_ids)."
    _show_images(ids or [i for i, in con.sql("SELECT image_id FROM images ORDER BY image_id").fetchall()])

def progress():
    "Show what you have submitted so far."
    rows = []
    for qid, q in QUESTIONS.items():
        a = work["answers"].get(qid, {})
        chk = a.get("check")
        rows.append({"question": qid, "": q["label"], "submissions": a.get("submissions", 0),
                     "last check": {"ok": "pass", "bad": "fail"}.get(chk["status"], "") if chk else "",
                     "written answer": "yes" if a.get("answer") else ""})
    display(pd.DataFrame(rows))
    print(f"{len(work['history'])} entries in the history.")

print(f"DuckDB {duckdb.__version__} is ready. Fitbit Air tables: {', '.join(TABLE_INFO)}; images: {len(IMAGE_INFO)}.")
print("OpenAI:", f"key found; Flock models embed ({EMBEDDING_MODEL}) and luna ({OPENAI_MODEL})"
      if OPENAI_API_KEY else "no key. Section 2 runs without one; section 3 needs OPENAI_API_KEY in Colab's Secrets panel.")
if work["history"]:
    print(f"Restored your saved work: {len(work['history'])} history entries. Tables you created are not restored.")

## 1 · The data

One person wore a Fitbit Air for four weeks, from Sep 14 to Oct 11, 2026. Each table follows a Google Health API
data type: **samples** (one reading at a time), **intervals** (a count over a time span), **sessions** (sleep and
exercise, with a start and an end), and **daily** summaries. The band charges every morning and was left off one
night, so some readings are missing. The person runs three evenings a week. All of it is synthetic.

In [ ]:
show_schema()

In [ ]:
show_table("heart_rate")

In [ ]:
run("""
SELECT * FROM sleep_stages
WHERE sleep_id = 'S01'
ORDER BY start_time
LIMIT 8;
""")

## 2 · Three time-series analyses

```sql
time_bucket(INTERVAL 1 HOUR, t)         -- tumbling window: the hour a timestamp falls in
avg(x) OVER (ORDER BY t ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)   -- sliding window: the 7 rows before this one
a ASOF JOIN b ON a.t >= b.t             -- each row of a, with the latest row of b at or before it
```

**Question 2.1 · Hourly windows.** The query computes each hour's mean heart rate on Oct 1 with `time_bucket` (a tumbling window). Add the number of readings in each hour. A fully worn hour has 720 readings, one every 5 seconds. Which hours have fewer, and why?

<details><summary>Hint</summary>

Add `count(*) AS n` to the `SELECT` list. The band records nothing while it charges or is off the wrist.
</details>

In [ ]:
submit("2.1", """
SELECT time_bucket(INTERVAL 1 HOUR, sample_time) AS hour,
       round(avg(bpm), 1) AS mean_bpm
FROM heart_rate
WHERE sample_time >= TIMESTAMP '2026-10-01' AND sample_time < TIMESTAMP '2026-10-02'
GROUP BY hour
ORDER BY hour;
""")

In [ ]:
explain("2.1", """

""")   # Which hours are short, and how should an hourly mean be reported for them?

**Question 2.2 · Personal baseline.** Each day's `baseline` is the mean resting heart rate of the 7 days before it: the window `w` slides one day at a time. Add `z`, how many standard deviations the day is from its baseline, and keep only the days with |z| > 2. Which days stand out?

<details><summary>Hint</summary>

`(bpm - avg(bpm) OVER w) / stddev_samp(bpm) OVER w AS z`. `QUALIFY abs(z) > 2` filters rows on a window function's result, the way `HAVING` filters groups.
</details>

In [ ]:
submit("2.2", """
SELECT date, bpm,
       round(avg(bpm) OVER w, 1) AS baseline
FROM daily_resting_heart_rate
WINDOW w AS (ORDER BY date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)
ORDER BY date;
""")

In [ ]:
explain("2.2", """

""")   # The person was ill from Oct 2 to Oct 4. Why did Oct 4 not stand out, what is Sep 28, and do the other daily tables agree?

**Question 2.3 · Align two series.** Heart rate and sleep stages have no timestamps in common. `ASOF JOIN` gives each heart-rate reading the stage that started at or before it, and `WHERE` keeps only readings before that stage ended. Change the query to return each stage's mean heart rate and number of readings.

<details><summary>Hint</summary>

Replace the `SELECT` list with `s.stage, round(avg(h.bpm), 1) AS mean_bpm, count(*) AS n`, remove `LIMIT 10`, and add `GROUP BY s.stage`.
</details>

In [ ]:
submit("2.3", """
SELECT h.sample_time, h.bpm, s.stage
FROM heart_rate AS h ASOF JOIN sleep_stages AS s ON h.sample_time >= s.start_time
WHERE h.sample_time < s.end_time
LIMIT 10;
""")

In [ ]:
explain("2.3", """

""")   # How does heart rate differ by sleep stage, and what would go wrong without the WHERE condition?

## 3 · Image search

The ten images are openly licensed photos from Wikimedia Commons (credits below), copied to the course site. `describe_image(path)` asks the
vision model to describe an image in one or two sentences; `embed(text)` is the Week 6 shortcut for
`llm_embedding(...)::FLOAT[1536]`. Under each result, `submit` prints the requests and tokens used.

In [ ]:
show_images()

In [ ]:
run("""
SELECT image_id, credit, license, source FROM images ORDER BY image_id;
""")

**Question 3.1 · Describe the images.** OpenAI's embedding models take text only, so a vision model first describes each image, and the description is embedded. Run the first cell to describe one image. Then, in the second cell, describe every image into `image_captions`.

<details><summary>Hint</summary>

Replace `...` with `describe_image(path)`.
</details>

In [ ]:
submit("3.1", """
SELECT image_id, describe_image(path) AS caption
FROM images
WHERE image_id = 'meal_pancakes';
""")

In [ ]:
submit("3.1", """
CREATE OR REPLACE TABLE image_captions AS
SELECT image_id, ... AS caption
FROM images;
""")

In [ ]:
explain("3.1", """

""")   # Which captions miss or invent a health-related detail?

**Question 3.2 · Search with text.** Embed each caption into `image_vectors`, as you embedded the notes in Week 6. Then search the images with text in the second cell, and try other searches: “a skin problem”, “an eye exam”, “a meal for someone watching their blood sugar”.

<details><summary>Hint</summary>

In the first cell, replace `...` with `embed(caption)`. The second cell is the Week 6 vector search, over images.
</details>

In [ ]:
submit("3.2", """
CREATE OR REPLACE TABLE image_vectors AS
SELECT image_id, ... AS vec
FROM image_captions;
""")

In [ ]:
submit("3.2", """
SET VARIABLE q = (SELECT embed('a breakfast high in sugar'));

SELECT v.image_id, c.caption, array_cosine_similarity(v.vec, getvariable('q')) AS similarity
FROM image_vectors AS v JOIN image_captions AS c USING (image_id)
ORDER BY similarity DESC
LIMIT 3;
""")

In [ ]:
explain("3.2", """

""")   # Which searches worked, and which did not? Why?

**Question 3.3 · Search with an image.** Find the 3 images most similar to the normal retinal photo, `fundus_normal`, using its stored vector instead of a text query. Fill in `WHERE`: `a` is `fundus_normal`, and `b` is any other image.

<details><summary>Hint</summary>

`WHERE a.image_id = 'fundus_normal' AND b.image_id <> a.image_id`.
</details>

In [ ]:
submit("3.3", """
SELECT b.image_id, array_cosine_similarity(a.vec, b.vec) AS similarity
FROM image_vectors AS a, image_vectors AS b
WHERE ...
ORDER BY similarity DESC
LIMIT 3;
""")

In [ ]:
explain("3.3", """

""")   # The search compares captions, not pixels. What could two retinal photos differ in that their captions would not show?

## 4 · Submit your notebook

Submit this notebook with its outputs: in Colab, **File > Download > Download .ipynb**, then upload that file.
The outputs under each cell (your results and the checks) are part of what is graded, so run every `submit` and
`explain` cell you edited before you download. `progress()` lists which questions you have submitted and
answered in words.

In [ ]:
progress()